# QC Plastik Biru Gulungan - YOLOv8 `black_speck`
**Sebelum mulai:** Runtime > Change runtime type > **GPU (T4)**.

Upload ke Google Drive: `MyDrive/qc_plastik/dataset_raw.zip` berisi file `.jpg` + `.txt` label YOLO (gambar bersih tanpa `.txt` / `.txt` kosong = negative sample).

Foto sintetis (nama file berawalan `synth_`) hanya masuk ke **train**. **Validasi hanya dari foto asli**; tambahkan foto asli dari kamera ke zip.

## 1. Mount Drive & install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q -U ultralytics
import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'GPU belum aktif!'

## 2. Siapkan dataset (ekstrak, validasi, split train/val 80/20)

In [ ]:
import zipfile, shutil, random
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/qc_plastik')
RAW = Path('/content/dataset_raw')
OUT = Path('/content/dataset')
VAL_RATIO, SEED = 0.2, 42

for d in (RAW, OUT):
    if d.exists(): shutil.rmtree(d)
RAW.mkdir(parents=True)
assert (DRIVE_DIR / 'dataset_raw.zip').exists(), (
    f'Tidak menemukan {DRIVE_DIR}/dataset_raw.zip. Pastikan folder qc_plastik ada di My Drive '
    '(bukan di dalam folder lain) dan zip sudah selesai di-upload.')
with zipfile.ZipFile(DRIVE_DIR / 'dataset_raw.zip') as z:
    z.extractall(RAW)

imgs = sorted(p for p in RAW.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
assert imgs, 'Tidak ada gambar di zip'

def has_boxes(p):
    return p.exists() and p.read_text().strip() != ''

def validate(p):
    if not p.exists(): return
    for n, line in enumerate(p.read_text().splitlines(), 1):
        if not line.strip(): continue
        v = line.split()
        assert len(v) == 5, f'{p}:{n} harus 5 nilai'
        assert int(v[0]) == 0, f'{p}:{n} class id harus 0'
        assert all(0 <= float(x) <= 1 for x in v[1:]), f'{p}:{n} koordinat harus ternormalisasi 0..1'

for im in imgs:
    validate(im.with_suffix('.txt'))

# Foto sintetis (nama berawalan synth_) HANYA dipakai untuk training.
# Validasi memakai foto asli dari kamera, supaya mAP mencerminkan kondisi nyata.
is_synth = lambda p: p.name.startswith('synth_')
real  = [p for p in imgs if not is_synth(p)]
synth = [p for p in imgs if is_synth(p)]
real_pos = [p for p in real if has_boxes(p.with_suffix('.txt'))]
real_neg = [p for p in real if not has_boxes(p.with_suffix('.txt'))]
print(f'Foto asli: {len(real)} (speck: {len(real_pos)}, bersih: {len(real_neg)}) | foto sintetis: {len(synth)}')

rng = random.Random(SEED)
def split(items):
    rng.shuffle(items)
    nv = max(1, round(len(items) * VAL_RATIO)) if len(items) > 1 else 0
    return items[nv:], items[:nv]

(pt, pv), (nt, nv) = split(real_pos), split(real_neg)
train, val = pt + nt + synth, pv + nv
if not val:
    print('PERINGATAN: belum ada foto asli. 10% data sintetis dipakai sebagai validasi, '
          'sehingga mAP TIDAK mencerminkan performa di mesin. Tambahkan foto asli ke zip.')
    rng.shuffle(synth)
    k = max(1, len(synth) // 10)
    val, train = synth[:k], synth[k:]
assert any(has_boxes(p.with_suffix('.txt')) for p in train), 'Belum ada gambar berlabel di train'

for s, files in {'train': train, 'val': val}.items():
    (OUT/'images'/s).mkdir(parents=True); (OUT/'labels'/s).mkdir(parents=True)
    for im in files:
        shutil.copy2(im, OUT/'images'/s/im.name)
        dst = OUT/'labels'/s/(im.stem + '.txt')
        src = im.with_suffix('.txt')
        shutil.copy2(src, dst) if has_boxes(src) else dst.write_text('')
    print(s, len(files))

DATA_YAML = OUT / 'data.yaml'
DATA_YAML.write_text(f'''path: {OUT.as_posix()}
train: images/train
val: images/val

nc: 1
names:
  0: black_speck
''')
print(DATA_YAML.read_text())

## 3. Training (yolov8n, 50 epoch)
Ukuran gambar diatur lewat `IMGSZ` (default 640). Pada dataset sintetis, 64% speck hanya berukuran < 4 piksel di 640; di 1280 tinggal 5%. Jika speck Anda kecil, ubah `IMGSZ = 1280` (training lebih lama ~3-4x). Gunakan nilai yang sama saat inferensi (`--imgsz`).

In [ ]:
from ultralytics import YOLO
RUN = 'black_speck_v1'
IMGSZ = 640   # ganti ke 1280 jika speck sangat kecil
BATCH = 16 if IMGSZ <= 640 else 8
model = YOLO('yolov8n.pt')
model.train(
    data=str(DATA_YAML), epochs=50, imgsz=IMGSZ, batch=BATCH, device=0, workers=2,
    project='/content/runs', name=RUN, exist_ok=True, seed=0,
    flipud=0.5, fliplr=0.5, degrees=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, mosaic=1.0, plots=True,
)

## 4. Evaluasi & simpan `best.pt` ke Drive

In [ ]:
best = Path('/content/runs')/RUN/'weights'/'best.pt'
m = YOLO(str(best)).val(data=str(DATA_YAML), imgsz=IMGSZ)
print(f'mAP50={m.box.map50:.4f} | mAP50-95={m.box.map:.4f} | P={m.box.mp:.4f} | R={m.box.mr:.4f}')

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, DRIVE_DIR/'best.pt')
shutil.copytree(Path('/content/runs')/RUN, DRIVE_DIR/'runs'/RUN, dirs_exist_ok=True)
print('Tersimpan:', DRIVE_DIR/'best.pt')

## 5. Uji cepat pada gambar val (opsional)

In [ ]:
import glob
from IPython.display import display
from PIL import Image
model = YOLO(str(DRIVE_DIR/'best.pt'))
for f in sorted(glob.glob(str(OUT/'images/val/*')))[:6]:
    r = model.predict(f, imgsz=IMGSZ, conf=0.35, verbose=False)[0]
    display(Image.fromarray(r.plot()[..., ::-1]))
    print(Path(f).name, '->', len(r.boxes), 'deteksi')